<a href="https://colab.research.google.com/github/HST0077/HYOTC/blob/main/Adjoint_Algorithmic_Differentiation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Python Class 객체**

In [3]:
class Scalar:
    def __init__(self, value):
        self.value = value

def add_two(obj):
    # 객체의 속성을 직접 수정
    obj.value += 2

In [4]:
# 실행
a = Scalar(10) # a는 실제 데이터를 가진 객체를 가리킴
print(f"초기값 a: {a.value}")

초기값 a: 10


In [6]:
add_two(a) # 함수에 객체(참조)를 전달
print(f"1회 호출 후 a: {a.value}")

1회 호출 후 a: 14


$f(a, b) = a(a + b)$ 라는 함수에서 $\left. \frac{\partial f}{\partial a} \right|_{a=2, b=3}$, $\left. \frac{\partial f}{\partial b} \right|_{a=2, b=3}$ 값을 구하여라.

In [2]:
from sympy import diff, symbols, exp

a,b=symbols('a b')
f=a*(a+b)
diff(f,a) # ∂f/∂a

2*a + b

In [3]:
diff(f, a).subs({a: 2, b: 3}).evalf()

7.00000000000000

In [4]:
diff(f,b) # ∂f/∂b

a

In [5]:
diff(f, b).subs({a: 2, b: 3}).evalf()

2.00000000000000

In [27]:
class Scalar:
    def __init__(self, value):
        self.value = value  # 값을 담는 그릇 (Value)
        self.adjoint = 0.0  # 미분값을 담는 그릇 (Adjoint)
        self.grad_fn = None # 이 그릇이 만들어진 '설계도(연결 고리)'

    def __add__(self, other):
        # 덧셈 그릇 설계
        out = Scalar(self.value + other.value)
        def backward():
            self.adjoint += out.adjoint * 1.0
            other.adjoint += out.adjoint * 1.0
        out.grad_fn = backward
        return out

    def __mul__(self, other):
        # 곱셈 그릇 설계
        out = Scalar(self.value * other.value)
        def backward():
            self.adjoint += out.adjoint * other.value
            other.adjoint += out.adjoint * self.value
        out.grad_fn = backward
        return out

# 1. 입력 그릇 준비 (a=2, b=3)
a = Scalar(2.0)
b = Scalar(3.0)

# 2. 계산: 이 줄들이 실행될 때 미분 설계도(grad_fn)가 자동으로 그려짐
w1 = a + Scalar(0.0) # w1이 덧셈에 의해 미분이 정의되도록 0을 더해줌
w2 = a + b
f = w1 * w2

# 3. 미분 에너지 주입 및 전파 (Adjoint Pass)
f.adjoint = 1.0         # 가장 마지막 그릇에 에너지 1을 채움

# 설계도를 거꾸로 따라가며 전파 (Tape에서 Pop하는 과정과 동일)
f.grad_fn()     # f -> w1, w2로 전파
w1.grad_fn()    # w1 -> a로 전파
w2.grad_fn()    # w2 -> a,b로 전파

print(f"--- 결과 확인 ---")
print(f"최종 값 f: {f.value:.4f}")
print(f"a의 편미분값(grad_a): {a.adjoint:.4f}")
print(f"b의 편미분값(grad_b): {b.adjoint:.4f}")


--- 결과 확인 ---
최종 값 f: 10.0000
a의 편미분값(grad_a): 7.0000
b의 편미분값(grad_b): 2.0000


$f(a, b) = a^3b^2$ 라는 함수에서 $\left. \frac{\partial f}{\partial a} \right|_{a=2, b=3}$, $\left. \frac{\partial f}{\partial b} \right|_{a=2, b=3}$ 값을 구하여라.

In [2]:
from sympy import diff, symbols, exp

a,b=symbols('a b')
f=a**3*b**2
diff(f,a) # ∂f/∂a

3*a**2*b**2

In [3]:
diff(f,b) # ∂f/∂b

2*a**3*b

In [4]:
diff(f, a).subs({a: 2, b: 3}).evalf()

108.000000000000

In [5]:
diff(f, b).subs({a: 2, b: 3}).evalf()

48.0000000000000

In [6]:
class Scalar:
    def __init__(self, value):
        self.value = value  # 값을 담는 그릇 (Value)
        self.adjoint = 0.0  # 미분값을 담는 그릇 (Adjoint)
        self.grad_fn = None # 이 그릇이 만들어진 '설계도(연결 고리)'

    def __add__(self, other):
        # 덧셈 그릇 설계
        out = Scalar(self.value + other.value)
        def backward():
            self.adjoint += out.adjoint * 1.0
            other.adjoint += out.adjoint * 1.0
        out.grad_fn = backward
        return out

    def __mul__(self, other):
        # 곱셈 그릇 설계
        out = Scalar(self.value * other.value)
        def backward():
            self.adjoint += out.adjoint * other.value
            other.adjoint += out.adjoint * self.value
        out.grad_fn = backward
        return out

In [9]:
# 1. 입력 그릇 준비 (a=2, b=3)
a = Scalar(2.0)
b = Scalar(3.0)

# 2. 계산: f = a^3 * b^2
# step 1: a^3 만들기
a2 = a * a      # w1 역할
a3 = a2 * a     # w2 역할 (a^3)

# step 2: b^2 만들기
b2 = b * b      # w3 역할 (b^2)

# step 3: 최종 결합
f = a3 * b2     # f = a^3 * b^2

# 3. 미분 에너지 주입 및 전파 (Adjoint Pass)
f.adjoint = 1.0         # 가장 마지막 그릇에 에너지 1을 채움

# 기록된 순서의 역순으로 터뜨리기
f.grad_fn()     # f -> a3, b2 에 에너지 전달
b2.grad_fn()    # b2 -> b, b 에 에너지 전달
a3.grad_fn()    # a3 -> a2, a 에 에너지 전달
a2.grad_fn()    # a2 -> a, a 에 에너지 전달

print(f"--- 결과 확인 ---")
print(f"최종 값 f: {f.value:.4f}")
print(f"a의 편미분값(grad_a): {a.adjoint:.4f}")
print(f"b의 편미분값(grad_b): {b.adjoint:.4f}")


--- 결과 확인 ---
최종 값 f: 72.0000
a의 편미분값(grad_a): 108.0000
b의 편미분값(grad_b): 48.0000


In [12]:
# pow 연산자 함수 사용
class Scalar:
    def __init__(self, value):
        self.value = value  # 값을 담는 그릇 (Value)
        self.adjoint = 0.0  # 미분값을 담는 그릇 (Adjoint)
        self.grad_fn = None # 이 그릇이 만들어진 '설계도(연결 고리)'

    def __pow__(self, n):
        # n은 Scalar 객체가 아닌 일반 숫자(int, float)라고 가정합니다.
        out = Scalar(self.value ** n)

        def backward():
            # 거듭제곱 미분 공식: n * x^(n-1)
            # 여기에 out.adjoint(전달받은 에너지)를 곱해서 누적합니다.
            self.adjoint += out.adjoint * (n * (self.value ** (n - 1)))
        out.grad_fn = backward
        return out

    def __mul__(self, other):
        # 곱셈 그릇 설계
        out = Scalar(self.value * other.value)
        def backward():
            self.adjoint += out.adjoint * other.value
            other.adjoint += out.adjoint * self.value
        out.grad_fn = backward
        return out

In [17]:
# 1. 입력 그릇 준비
a = Scalar(2.0)
b = Scalar(3.0)

# 2. 계산 (Forward Pass)
term_a = a ** 3   # a^3 라는 중간 객체
term_b = b ** 2   # b^2 라는 중간 객체
f = term_a * term_b

# 3. 미분 에너지 주입 및 전파 (Adjoint Pass)
f.adjoint = 1.0

# 수식이 간결해진 만큼 grad_fn 호출 순서도 명확해집니다.
f.grad_fn()     # 곱셈 미분 전파 (a^3과 b^2로 분산)

term_a.grad_fn()  # 2. term_a가 받은 에너지를 a에게 배달함 (3a^2 공식 적용)
term_b.grad_fn()  # 3. term_b가 받은 에너지를 b에게 배달함 (2b 공식 적용)

print(f"--- 결과 확인 ---")
print(f"최종 값 f: {f.value:.4f}")
print(f"a의 편미분값(grad_a): {a.adjoint:.4f}")
print(f"b의 편미분값(grad_b): {b.adjoint:.4f}")

--- 결과 확인 ---
최종 값 f: 72.0000
a의 편미분값(grad_a): 108.0000
b의 편미분값(grad_b): 48.0000


$f(a, b) = (a + b)e^a$ 라는 함수에서 $\left. \frac{\partial f}{\partial a} \right|_{a=2, b=3}$, $\left. \frac{\partial f}{\partial b} \right|_{a=2, b=3}$ 값을 구하여라.

In [18]:
from sympy import diff, symbols, exp

a,b=symbols('a b')
f=(a+b)*exp(a)
diff(f,a) # ∂f/∂a

(a + b)*exp(a) + exp(a)

In [19]:
diff(f,b) # ∂f/∂b

exp(a)

In [20]:
diff(f, a).subs({a: 2, b: 3}).evalf()

44.3343365935839

In [21]:
diff(f, b).subs({a: 2, b: 3}).evalf()

7.38905609893065

In [23]:
import math

class Scalar:
    def __init__(self, value):
        self.value = value  # 값을 담는 그릇 (Value)
        self.adjoint = 0.0  # 미분값을 담는 그릇 (Adjoint)
        self.grad_fn = None # 이 그릇이 만들어진 '설계도(연결 고리)'

    def __add__(self, other):
        # 덧셈 그릇 설계
        out = Scalar(self.value + other.value)
        def backward():
            self.adjoint += out.adjoint * 1.0
            other.adjoint += out.adjoint * 1.0
        out.grad_fn = backward
        return out

    def __mul__(self, other):
        # 곱셈 그릇 설계
        out = Scalar(self.value * other.value)
        def backward():
            self.adjoint += out.adjoint * other.value
            other.adjoint += out.adjoint * self.value
        out.grad_fn = backward
        return out

# --- exp를 위한 '그릇 설계' 함수 ---
def exp_scalar(x: Scalar):
    # 1. Forward: 값 계산
    out = Scalar(math.exp(x.value))

    # 2. 설계도: exp(x)를 미분하면 exp(x)이므로,
    # out.value를 그대로 사용
    def backward():
        x.adjoint += out.adjoint * out.value

    out.grad_fn = backward
    return out



In [24]:
# --- 실제 실행 ---

# 1. 입력 그릇 준비 (a=2, b=3)
a = Scalar(2.0)
b = Scalar(3.0)

# 2. 계산: 이 줄들이 실행될 때 미분 설계도(grad_fn)가 자동으로 그려짐
w1 = a + b
w2 = exp_scalar(a)
f = w1 * w2

# 3. 미분 에너지 주입 및 전파 (Adjoint Pass)
f.adjoint = 1.0         # 가장 마지막 그릇에 에너지 1을 채움

# 설계도를 거꾸로 따라가며 전파 (Tape에서 Pop하는 과정과 동일)
f.grad_fn()     # f -> w1, w2로 전파
w2.grad_fn()    # w2 -> a로 전파
w1.grad_fn()    # w1 -> a, b로 전파

print(f"--- 결과 확인 ---")
print(f"최종 값 f: {f.value:.4f}")
print(f"a의 편미분값(grad_a): {a.adjoint:.4f}")
print(f"b의 편미분값(grad_b): {b.adjoint:.4f}")

--- 결과 확인 ---
최종 값 f: 36.9453
a의 편미분값(grad_a): 44.3343
b의 편미분값(grad_b): 7.3891


# 유럽형 콜옵션의 평가로직을 Monte Carlo로 구현하고, Greeks를 ADD 방식으로 구현해 보아라.

In [25]:
# BSM 공식
"""
Black-Scholes 해석 공식 (배당수익률 q 적용)
Plain/BSM_MC.py의 MC_Call, MC_Greeks와 동일 계약·Greeks 정의
"""

from math import log, sqrt, exp
from scipy.stats import norm


def _d1_d2(S, K, T, r, q, sigma):
    if T <= 0:
        return 0.0, 0.0
    sqrtT = sqrt(T)
    d1 = (log(S / K) + (r - q + 0.5 * sigma**2) * T) / (sigma * sqrtT)
    d2 = d1 - sigma * sqrtT
    return d1, d2


def BSM(S, K, T, r, q, sigma, option="call"):
    """
    European 옵션 해석 가격 (연속 배당수익률 q).
    option: "call" | "put"
    """
    opt = option.lower().strip()
    if opt not in ("call", "put"):
        raise ValueError('option must be "call" or "put"')
    if T <= 0:
        return max(S - K, 0.0) if opt == "call" else max(K - S, 0.0)
    d1, d2 = _d1_d2(S, K, T, r, q, sigma)
    if opt == "call":
        return S * exp(-q * T) * norm.cdf(d1) - K * exp(-r * T) * norm.cdf(d2)
    return K * exp(-r * T) * norm.cdf(-d2) - S * exp(-q * T) * norm.cdf(-d1)


# 하위호환
BSM_Call = lambda S, K, T, r, q, sigma: BSM(S, K, T, r, q, sigma, "call")
BSM_Put = lambda S, K, T, r, q, sigma: BSM(S, K, T, r, q, sigma, "put")


def BSM_Greeks(S, K, T, r, q, sigma, option="call"):
    """
    해석적 Greeks.
    option: "call" | "put"
    반환: Price, Delta(Δ), Gamma(Γ), Theta(1day), Vega(1%), Rho(1bp)
    """
    opt = option.lower().strip()
    if opt not in ("call", "put"):
        raise ValueError('option must be "call" or "put"')

    if T <= 0:
        intrinsic = max(S - K, 0.0) if opt == "call" else max(K - S, 0.0)
        delta_int = (1.0 if S > K else 0.0) if opt == "call" else (-1.0 if S < K else 0.0)
        return {
            "Price": intrinsic,
            "Delta (Δ)": delta_int,
            "Gamma (Γ)": 0.0,
            "Theta (Θ)": 0.0,
            "Vega (V)": 0.0,
            "Rho (ρ)": 0.0,
        }

    d1, d2 = _d1_d2(S, K, T, r, q, sigma)
    sqrtT = sqrt(T)
    n_d1 = norm.pdf(d1)

    # Price
    price = BSM(S, K, T, r, q, sigma, opt)
    if opt == "call":
        Delta = exp(-q * T) * norm.cdf(d1)
        Rho = 0.0001 * (T * K * exp(-r * T) * norm.cdf(d2))
    else:
        Delta = -exp(-q * T) * norm.cdf(-d1)  # = exp(-q*T)*(N(d1)-1)
        Rho = 0.0001 * (-T * K * exp(-r * T) * norm.cdf(-d2))

    # Gamma (Γ): Call과 Put 동일
    Gamma = exp(-q * T) * n_d1 / (S * sigma * sqrtT) if sqrtT > 0 and sigma > 0 else 0.0

    # Vega: Call과 Put 동일
    Vega = 0.01 * S * exp(-q * T) * sqrtT * n_d1

    # Theta (1day)
    eps_t = 1.0 / 365.0
    T_eps = max(T - eps_t, 1e-10)
    Theta = BSM(S, K, T_eps, r, q, sigma, opt) - price

    return {
        "Price": float(price),
        "Delta (Δ)": float(Delta),
        "Gamma (Γ)": float(Gamma),
        "Theta (Θ)": float(Theta),
        "Vega (V)": float(Vega),
        "Rho (ρ)": float(Rho),
    }


In [26]:
BSM_Greeks(100, 100, 1, 0.05, 0, 0.3,'call')

{'Price': 14.231254785985819,
 'Delta (Δ)': 0.6242517279060125,
 'Gamma (Γ)': 0.012647764437231512,
 'Theta (Θ)': -0.022207200555286022,
 'Vega (V)': 0.37943293311694537,
 'Rho (ρ)': 0.004819391800461543}

In [38]:
# MC+ADD approach

TAPE = []  # 모든 연산 노드를 순서대로 담는 전역 리스트

class Scalar:
    def __init__(self, value):
        self.value = value
        self.adjoint = 0.0
        self.grad_fn = None

    def __add__(self, other):
        other = other if isinstance(other, Scalar) else Scalar(other)
        out = Scalar(self.value + other.value)
        def backward():
            self.adjoint += out.adjoint
            other.adjoint += out.adjoint
        out.grad_fn = backward
        TAPE.append(out)  # 자동으로 테이프에 기록
        return out

    def __sub__(self, other):
        other = other if isinstance(other, Scalar) else Scalar(other)
        out = Scalar(self.value - other.value)
        def backward():
            self.adjoint += out.adjoint
            other.adjoint -= out.adjoint
        out.grad_fn = backward
        TAPE.append(out)
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Scalar) else Scalar(other)
        out = Scalar(self.value * other.value)
        def backward():
            self.adjoint += out.adjoint * other.value
            other.adjoint += out.adjoint * self.value
        out.grad_fn = backward
        TAPE.append(out)
        return out

    def __pow__(self, power):
        out = Scalar(self.value ** power)
        def backward():
            self.adjoint += out.adjoint * (power * self.value**(power - 1))
        out.grad_fn = backward
        TAPE.append(out)
        return out

# 수학 함수들도 TAPE에 기록하도록 수정
def exp_s(x):
    out = Scalar(math.exp(x.value))
    def backward():
        x.adjoint += out.adjoint * out.value
    out.grad_fn = backward
    TAPE.append(out)
    return out

def sqrt_s(x):
    out = Scalar(math.sqrt(x.value))
    def backward():
        x.adjoint += out.adjoint * (0.5 / out.value)
    out.grad_fn = backward
    TAPE.append(out)
    return out

In [43]:
import random
import math

def call_option_full_greeks_aad_fixed(S, K, T, r, q, vol, num_paths=20000):
    global TAPE
    TAPE = []  # 실행 전 테이프 초기화

    S_s, T_s, r_s, q_s, vol_s = Scalar(S), Scalar(T), Scalar(r), Scalar(q), Scalar(vol)

    path_payoffs = []
    for _ in range(num_paths):
        z = random.gauss(0, 1)

        # S * exp((r - q - 0.5*vol^2)*T + vol*sqrt(T)*z)
        vol_sq = vol_s * vol_s
        drift = (r_s - q_s - (vol_sq * 0.5)) * T_s
        diffusion = vol_s * sqrt_s(T_s) * z
        ST = S_s * exp_s(drift + diffusion)

        payoff = ST - K if ST.value > K else Scalar(0.0)
        path_payoffs.append(payoff)

    # 평균 및 할인
    avg_payoff = sum(path_payoffs, Scalar(0.0)) * (1.0 / num_paths)
    df = exp_s(r_s * T_s * -1.0)
    price_s = avg_payoff * df

    # --- Adjoint Pass (이 부분이 핵심입니다) ---
    price_s.adjoint = 1.0

    # 덧셈 결과인 avg_payoff 등은 이미 TAPE의 마지막 부분에 있습니다.
    # TAPE에 담긴 모든 노드를 생성된 역순으로 실행합니다.
    for node in reversed(TAPE):
        if node.grad_fn:
            node.grad_fn()

    return {
        "Price": price_s.value,
        "Delta": S_s.adjoint,
        "Vega(1%)": vol_s.adjoint * 0.01,
        "Rho(1bp)": r_s.adjoint * 0.0001,
        "Theta(1day)": (-T_s.adjoint) / 365.0
    }

In [42]:
# 실행 및 결과 확인
greeks = call_option_full_greeks_aad_fixed(100, 100, 1.0, 0.05, 0, 0.3)

print(f"{'Greek':<10} | {'Value':<10}")
print("-" * 25)
for name, val in greeks.items():
    print(f"{name:<10} | {val:>10.4f}")

Greek      | Value     
-------------------------
Price      |    14.0789
Delta      |     0.6223
Vega(1%)   |     0.3731
Rho(1bp)   |     0.0048
Theta(1day) |    -0.0219


In [30]:
BSM_Greeks(100, 100, 1, 0.05, 0, 0.3,'call')

{'Price': 14.231254785985819,
 'Delta (Δ)': 0.6242517279060125,
 'Gamma (Γ)': 0.012647764437231512,
 'Theta (Θ)': -0.022207200555286022,
 'Vega (V)': 0.37943293311694537,
 'Rho (ρ)': 0.004819391800461543}

In [44]:
def calculate_gamma_with_aad(S, K, T, r, q, vol, num_paths=20000):
    h = S * 0.01  # 주가의 1%만큼 변화

    # 1. S + h 일 때의 Delta 구하기
    result_plus = call_option_full_greeks_aad_fixed(S + h, K, T, r, q, vol, num_paths)
    delta_plus = result_plus["Delta"]

    # 2. S - h 일 때의 Delta 구하기
    result_minus = call_option_full_greeks_aad_fixed(S - h, K, T, r, q, vol, num_paths)
    delta_minus = result_minus["Delta"]

    # 3. Gamma 산출 (Delta의 변화율)
    gamma = (delta_plus - delta_minus) / (2 * h)

    return gamma

In [45]:
calculate_gamma_with_aad(100, 100, 1, 0.05, 0, 0.3, num_paths=20000)

0.010298983307209741